# Bank Marketing: Exploratory Data Analysis

**Goal of this notebook:** understand the data before building the case-based reasoning (CBR) system.

Each row is a **client** contacted by phone by a bank. The target `y` says whether he **subscribed to a term deposit** after the call.

What we will do:
1. Load the data and look at its structure
2. Check data quality (missing values, `unknown`, duplicates, special codes)
3. Study the **target** (it is very imbalanced)
4. Study each **variable** (numeric then categorical)
5. Study the **relationship between each variable and the target**
6. Rank variables by usefulness (helps choose **similarity weights** for CBR)
7. Summarise what it means for the CBR design

## 1. Imports and settings

In [7]:
from pathlib import Path 
try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:  # running inside a notebook
    ROOT = Path.cwd()

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
RAW_FILE = "bank-additional-full.csv"
print(RAW_DIR)

c:\Users\pc\Desktop\Master-TPS\M2\ML fundamentals\CBR & RAG\notebooks\data\raw


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4)
pd.set_option("display.max_columns", 50)

## 2. Load the data

Two options:
- `"uci"`: fetch with `ucimlrepo` (the code you gave)
- `"csv"`: read the local file `bank-additional-full.csv` (separator is `;`)

> **Important:** `fetch_ucirepo(id=222)` returns the **older version** of the dataset (`bank-full`: about 45,211 rows, 16 variables, with `balance`, no economic indicators).
> Your assignment uses **`bank-additional-full.csv`** (41,188 rows, 20 variables, with `euribor3m`, `emp.var.rate`, ...).
> Download the CSV from the UCI page (it is inside `bank+marketing.zip` > `bank-additional`) and switch to `"csv"` for your real work.
> Everything below works with both versions.

In [ ]:
DATA_SOURCE = "uci"          # "uci" or "csv"
CSV_PATH = r"bank-additional-full.csv"

if DATA_SOURCE == "uci":
    from ucimlrepo import fetch_ucirepo
    bank_marketing = fetch_ucirepo(id=222)
    X = bank_marketing.data.features
    y = bank_marketing.data.targets
    df = X.copy()
    df["y"] = y.iloc[:, 0].values
else:
    df = pd.read_csv(CSV_PATH, sep=";")

print("Shape:", df.shape)

In [ ]:
# Metadata and variable descriptions (only available with the UCI fetch)
if DATA_SOURCE == "uci":
    print(bank_marketing.metadata["abstract"])
    print("Instances:", bank_marketing.metadata["num_instances"])
    display(bank_marketing.variables[["name", "role", "type", "description", "missing_values"]])

## 3. First look

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Summary statistics of numeric variables
df.describe().T

## 4. Data quality

We check four things: **missing values**, **`unknown` values**, **duplicates**, and **special codes** (like `pdays`).

In [ ]:
# 4.1 Real missing values (NaN) and the 'unknown' category
nan_count = df.isna().sum()
unknown_count = (df == "unknown").sum()
quality = pd.DataFrame({"NaN": nan_count, "'unknown'": unknown_count})
quality["total missing-like"] = quality.sum(axis=1)
quality["% of rows"] = (quality["total missing-like"] / len(df) * 100).round(2)
quality[quality["total missing-like"] > 0].sort_values("% of rows", ascending=False)

The UCI fetch uses `NaN` for missing categories, while the CSV writes `unknown`. To treat both versions the same way, we turn `NaN` into `unknown`. For CBR, `unknown` can simply be treated as one more category.

In [ ]:
cat_raw = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c != "y"]
df[cat_raw] = df[cat_raw].fillna("unknown")

# Binary target: 1 = subscribed, 0 = did not
df["target"] = (df["y"] == "yes").astype(int)

TARGET = "target"
num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET]
cat_cols = [c for c in df.columns if c not in num_cols + ["y", TARGET]]

print("Numeric variables    :", num_cols)
print("Categorical variables:", cat_cols)

In [ ]:
# 4.2 Duplicates
print("Duplicated rows:", df.duplicated().sum())

A few duplicated rows are not necessarily errors: two different clients can have identical attributes. We keep them, but it is worth mentioning.

## 5. The target variable

Only a minority of clients subscribe. This is the **most important fact** of the dataset: it is why we use **balanced accuracy** and not accuracy.

In [ ]:
counts = df["y"].value_counts()
percent = (df["y"].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({"count": counts, "%": percent}))

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x="y", order=["no", "yes"], ax=ax)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
ax.set_title("Target distribution")
plt.show()

majority = counts.max() / len(df)
print(f"A model that always answers 'no' gets accuracy = {majority:.3f}")
print("...but its balanced accuracy is only 0.500 (it never finds a 'yes').")

## 6. Numeric variables: distributions

Questions to look for: is the variable skewed? are there outliers? are there special values (like 999)?

In [ ]:
n = len(num_cols); ncols = 3; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.4 * nrows))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c], bins=40, ax=ax)
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Skewness: values far from 0 mean a strongly asymmetric distribution
df[num_cols].skew().sort_values(ascending=False).round(2).to_frame("skewness")

In [ ]:
# Boxplots reveal outliers
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.0 * nrows))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(x=df[c], ax=ax)
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Special numeric variables

- **`pdays`**: days since the client was last contacted in a *previous* campaign. A special code means "never contacted" (`999` in the additional version, `-1` in the older one). It is **not a real number of days**, so using it directly in a distance would be wrong.
- **`previous`**: number of contacts *before* this campaign.
- **`campaign`**: number of contacts during *this* campaign (including this call). It is known before the call, so we keep it.
- **`duration`**: length of the call. **Not known before the call**, so it must be removed (data leakage).

In [ ]:
if "pdays" in df:
    never = df["pdays"].isin([-1, 999])
    print(f"Never contacted before: {never.mean():.1%} of clients")
    df["previously_contacted"] = (~never).astype(int)

    display(df.groupby("previously_contacted")[TARGET].agg(rate="mean", count="size").round(3))

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    sns.histplot(df.loc[~never, "pdays"], bins=40, ax=axes[0])
    axes[0].set_title("pdays, only previously contacted clients")
    sns.barplot(data=df, x="previously_contacted", y=TARGET, ax=axes[1])
    axes[1].set_title("Subscription rate by 'previously contacted'")
    plt.tight_layout(); plt.show()

In [ ]:
# 'campaign' is heavy-tailed: group the large values to read it better
camp = df["campaign"].clip(upper=10)
rate_camp = df.groupby(camp)[TARGET].agg(rate="mean", count="size")
rate_camp.index = [str(i) if i < 10 else "10+" for i in rate_camp.index]

fig, ax = plt.subplots(figsize=(8, 3.8))
sns.barplot(x=rate_camp.index, y=rate_camp["rate"], ax=ax)
ax.axhline(df[TARGET].mean(), color="red", ls="--", label="overall rate")
ax.set_xlabel("calls during this campaign"); ax.set_ylabel("subscription rate"); ax.legend()
plt.show()

In [ ]:
# Why 'duration' must be removed: it is strongly linked to the target, but unknown before the call
print(df.groupby("y")["duration"].describe().round(1))

dur_bins = pd.qcut(df["duration"], 10, duplicates="drop")
dur_rate = df.groupby(dur_bins)[TARGET].mean()
fig, ax = plt.subplots(figsize=(9, 3.8))
dur_rate.plot(kind="bar", ax=ax)
ax.set_title("Subscription rate by duration decile (this is leakage)")
ax.set_ylabel("rate"); ax.set_xlabel("")
plt.xticks(rotation=45, ha="right"); plt.show()

# Dataset used from now on for modelling
df_model = df.drop(columns=["duration"])
num_model = [c for c in num_cols if c != "duration"]
print("duration removed. Remaining variables:", df_model.shape[1] - 2)

## 7. Categorical variables: distributions

In [ ]:
display(pd.DataFrame({
    "n_categories": df[cat_cols].nunique(),
    "most frequent": [df[c].value_counts().index[0] for c in cat_cols],
    "its share (%)": [round(df[c].value_counts(normalize=True).iloc[0] * 100, 1) for c in cat_cols],
}))

In [ ]:
n = len(cat_cols); ncols = 3; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.8 * nrows))
for ax, c in zip(axes.ravel(), cat_cols):
    order = df[c].value_counts().index
    sns.countplot(data=df, y=c, order=order, ax=ax)
    ax.set_title(c); ax.set_ylabel("")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

Watch for: categories that are almost the whole dataset (`default`: almost nobody has defaulted, so it carries little information), and rare categories.

## 8. Each variable versus the target

### 8.1 Numeric variables
Do clients who subscribe look different from those who do not?

In [ ]:
display(df_model.groupby("y")[num_model].mean().T.round(2))

n = len(num_model); ncols = 3; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.4 * nrows))
for ax, c in zip(axes.ravel(), num_model):
    sns.boxplot(data=df_model, x="y", y=c, order=["no", "yes"], showfliers=False, ax=ax)
    ax.set_title(c); ax.set_xlabel("")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### 8.2 Categorical variables
For each category we plot the **subscription rate** (and the number of clients). The red line is the overall rate: bars above it mean "more likely to subscribe than average".
Beware of categories with very few clients: their rate is unreliable.

In [ ]:
overall = df[TARGET].mean()
n = len(cat_cols); ncols = 3; nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.8 * nrows))
for ax, c in zip(axes.ravel(), cat_cols):
    t = df.groupby(c)[TARGET].agg(rate="mean", count="size").sort_values("rate")
    sns.barplot(x=t["rate"], y=t.index, ax=ax)
    ax.axvline(overall, color="red", ls="--")
    for i, (r, k) in enumerate(zip(t["rate"], t["count"])):
        ax.text(r, i, f" n={k}", va="center", fontsize=8)
    ax.set_title(c); ax.set_ylabel(""); ax.set_xlabel("subscription rate")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Previous campaign outcome is usually one of the strongest signals
if "poutcome" in df:
    display(df.groupby("poutcome")[TARGET].agg(rate="mean", count="size").round(3))

In [ ]:
# Time effect: month (campaigns are not spread evenly over the year)
if "month" in df:
    order = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
    order = [m for m in order if m in df["month"].unique()]
    t = df.groupby("month")[TARGET].agg(rate="mean", count="size").loc[order]

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    sns.barplot(x=t.index, y=t["count"], ax=axes[0]); axes[0].set_title("Number of calls per month")
    sns.barplot(x=t.index, y=t["rate"], ax=axes[1]); axes[1].set_title("Subscription rate per month")
    axes[1].axhline(overall, color="red", ls="--")
    plt.tight_layout(); plt.show()

## 9. Correlations between numeric variables

Strongly correlated variables carry the **same information twice**. In a similarity measure they would count double, so their weights should be reduced.

In [ ]:
corr = df_model[num_model + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix (numeric variables)")
plt.show()

print("Correlation with the target:")
display(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3).to_frame("corr"))

In [ ]:
# Pairs of highly correlated variables (|r| > 0.7)
c = corr.drop(index=TARGET, columns=TARGET)
pairs = [(a, b, round(c.loc[a, b], 2)) for i, a in enumerate(c.index) for b in c.columns[i + 1:] if abs(c.loc[a, b]) > 0.7]
pd.DataFrame(pairs, columns=["var 1", "var 2", "correlation"])

## 10. Which variables are most informative? (mutual information)

**Mutual information** measures how much knowing a variable tells you about the target. It works for numeric and categorical variables and also captures non-linear links. It gives a data-driven starting point for **the weights of your similarity function**.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_mi = df_model.drop(columns=["y", TARGET, "previously_contacted"], errors="ignore").copy()
discrete = []
for col in X_mi.columns:
    if pd.api.types.is_numeric_dtype(X_mi[col]):
        discrete.append(False)
    else:
        X_mi[col] = X_mi[col].astype("category").cat.codes
        discrete.append(True)

mi = mutual_info_classif(X_mi, df_model[TARGET], discrete_features=discrete, random_state=0)
mi = pd.Series(mi, index=X_mi.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=mi.values, y=mi.index, ax=ax)
ax.set_title("Mutual information with the target")
plt.show()
mi.round(4).to_frame("mutual information")

## 11. Local distance per variable (preparing the CBR)

In CBR, the global distance between two clients is a **weighted sum of local distances**, one per variable. The table below proposes a local distance for each variable according to its type.

In [ ]:
rows = []
for col in X_mi.columns:
    if col in num_model:
        kind = "numeric"
        rule = "|a - b| / (max - min)"
        if col == "pdays":
            rule = "handle the 'never contacted' code separately (flag), then |a - b| / range"
    else:
        kind = "categorical"
        rule = "0 if same category, 1 otherwise"
    rows.append({"variable": col, "type": kind,
                 "n unique": df_model[col].nunique(),
                 "local distance": rule,
                 "mutual info": round(mi[col], 4)})
pd.DataFrame(rows).sort_values("mutual info", ascending=False).reset_index(drop=True)

## 12. Summary: what the data tells us

Fill this section with your own observations once you run the notebook. Points to check:

1. **Imbalance**: about 11% of clients subscribe, so use **balanced accuracy** (and tune the decision threshold), never plain accuracy.
2. **Leakage**: `duration` must be removed; `campaign` can stay.
3. **`pdays`** contains a special code (999 or -1): build a `previously_contacted` flag and do not treat the code as a number of days.
4. **`unknown`** appears in several categorical variables: keep it as its own category.
5. **Strongest signals** (check the plots and the mutual information): outcome of the previous campaign (`poutcome`), economic indicators (if present), contact type, month.
6. **Redundant variables**: economic indicators are highly correlated with each other, so give them a lower individual weight.
7. **Weights for the similarity**: start from the mutual information ranking, justify them, then refine on a validation set.
8. **Split**: use a stratified train/test split, and tune on a validation part of the training data only.